# BSM implied volatility by bisection

시장가격 $V_{market}$에 대해 다음 scalar 방정식의 해를 찾는다.

$$
f(\sigma) = V_{BSM}(\sigma) - V_{market} = 0
$$

유럽형 옵션의 BSM 가격은 변동성에 대해 단조 증가하므로, 해가 설정한 bracket 안에 있으면 이분탐색으로 bracket을 반복해서 절반으로 줄일 수 있다. 구현은 기존 `bsm_price`를 재사용하며, 가격 오차 또는 변동성 bracket 폭이 각 tolerance 이하가 되면 수렴한다. 무차익 범위 밖 가격, bracket 안에서 만들 수 없는 가격, 반복 횟수 안에 수렴하지 못한 경우는 임의의 IV 대신 명시적으로 실패한다.

In [1]:
from __future__ import annotations

import math
from numbers import Real
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in project_root_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root from the current directory")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from option_pricing_volatility.models.bsm import bsm_price
from option_pricing_volatility.volatility import implied_volatility

## Synthetic round-trip

알려진 변동성으로 call과 put의 BSM 가격을 만든 뒤, 그 가격만을 target으로 사용해 IV를 복원한다. 마지막 열은 복원한 IV로 다시 계산한 BSM 가격에서 target 가격을 뺀 signed repricing error다.

In [2]:
synthetic_parameters = {
    "spot": 100.0,
    "strike": 105.0,
    "maturity": 0.75,
    "rate": 0.03,
    "dividend_yield": 0.01,
}
original_volatility = 0.24
price_tolerance = 1e-8

round_trip_rows = []
for option_type in ("call", "put"):
    market_price = bsm_price(
        **synthetic_parameters,
        volatility=original_volatility,
        option_type=option_type,
    )
    result = implied_volatility(
        **synthetic_parameters,
        market_price=market_price,
        option_type=option_type,
        price_tolerance=price_tolerance,
    )
    repriced = bsm_price(
        **synthetic_parameters,
        volatility=result.volatility,
        option_type=option_type,
    )
    round_trip_rows.append(
        {
            "option_type": option_type,
            "original_volatility": original_volatility,
            "recovered_implied_volatility": result.volatility,
            "volatility_error": result.volatility - original_volatility,
            "repricing_error": repriced - market_price,
        }
    )

round_trip_df = pd.DataFrame(round_trip_rows)
assert round_trip_df["volatility_error"].abs().max() <= 1e-8
assert round_trip_df["repricing_error"].abs().max() <= price_tolerance
round_trip_df

,option_type,original_volatility,recovered_implied_volatility,volatility_error,repricing_error
0,call,0.24,0.24,-3.536954e-11,-1.210786e-09
1,put,0.24,0.24,-3.536954e-11,-1.210779e-09


## Fixed SPX processed snapshot

고정된 processed snapshot의 유효한 `bid`와 `ask`에서 이미 계산된 `mid`를 시장가격으로 사용한다. `last`와 provider의 IV는 대체값으로 사용하지 않는다.

IV 계산에는 snapshot과 같은 기준시점의 연속복리 `risk_free_rate`와 `dividend_yield`가 필요하다. 현재 processed 파일에는 두 열과 그 출처가 없으므로 이 Notebook은 값을 임의로 가정하지 않는다. 현재 행은 삭제하지 않고 `NaN`과 `MISSING_MODEL_INPUT` 사유로 보존한다. 향후 출처가 기록된 두 열이 processed 입력에 추가되면 아래 동일한 행별 코드가 IV를 계산한다. `forward`도 새로 추정하지 않으며, 입력에 이미 존재할 때만 $\log(K/F)$를 사용한다.

In [3]:
SPX_PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "marketdata_spx"
    / "SPX_2026-07-15_dte030_pm_sl450_processed.csv"
)
if not SPX_PROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"Expected the existing processed SPX snapshot at {SPX_PROCESSED_PATH}"
    )

spx_df = pd.read_csv(SPX_PROCESSED_PATH)
required_columns = {
    "underlying",
    "option_type",
    "strike",
    "spot",
    "T",
    "bid",
    "ask",
    "mid",
}
missing_columns = sorted(required_columns.difference(spx_df.columns))
if missing_columns:
    raise ValueError(f"Processed SPX schema is missing columns: {missing_columns}")
if not spx_df["underlying"].eq("SPX").all():
    raise ValueError("Processed snapshot contains a non-SPX underlying")
if not ((spx_df["bid"] > 0.0) & (spx_df["ask"] >= spx_df["bid"])).all():
    raise ValueError("Processed snapshot contains an invalid two-sided quote")
if not np.allclose(
    spx_df["mid"],
    (spx_df["bid"] + spx_df["ask"]) / 2.0,
    rtol=0.0,
    atol=1e-12,
):
    raise ValueError("mid does not match the bid/ask midpoint convention")

spx_df = spx_df.copy()
spx_df["target_price"] = spx_df["mid"]
spx_df["market_price"] = spx_df["target_price"]
spx_df.shape, spx_df[["option_type", "strike", "market_price"]].head()

((424, 36),
   option_type  strike  market_price
 0        call    2400       5175.85
 1        call    2600       4976.95
 2        call    2800       4777.30
 3        call    3000       4579.00
 4        call    3200       4378.60)

In [4]:
model_input_columns = ("risk_free_rate", "dividend_yield")
missing_model_columns = [
    column for column in model_input_columns if column not in spx_df.columns
]

def invert_spx_row(row: pd.Series) -> pd.Series:
    if missing_model_columns:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": (
                    "MISSING_MODEL_INPUT:" + ",".join(missing_model_columns)
                ),
            }
        )

    nonfinite_inputs = [
        column
        for column in model_input_columns
        if (
            isinstance(row[column], (bool, np.bool_))
            or not isinstance(row[column], Real)
            or not math.isfinite(row[column])
        )
    ]
    if nonfinite_inputs:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": (
                    "NONFINITE_MODEL_INPUT:" + ",".join(nonfinite_inputs)
                ),
            }
        )

    try:
        result = implied_volatility(
            spot=row["spot"],
            strike=row["strike"],
            maturity=row["T"],
            rate=row["risk_free_rate"],
            market_price=row["market_price"],
            option_type=row["option_type"],
            dividend_yield=row["dividend_yield"],
        )
    except (ValueError, RuntimeError) as exc:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": f"{type(exc).__name__}: {exc}",
            }
        )

    return pd.Series(
        {
            "implied_volatility": result.volatility,
            "repricing_error": result.repricing_error,
            "iv_iterations": result.iterations,
            "iv_converged": result.converged,
            "iv_status": "success",
            "iv_failure_reason": pd.NA,
        }
    )

iv_results = spx_df.apply(invert_spx_row, axis=1)
spx_iv_df = pd.concat([spx_df, iv_results], axis=1)
iv_run_summary = (
    spx_iv_df.groupby(["iv_status", "iv_failure_reason"], dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
)
iv_run_summary

,iv_status,iv_failure_reason,row_count
0,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield",424


## Plotting-ready tidy result

기존 `forward`가 있으면 `log_forward_moneyness = log(strike / forward)`를 계산하고 그 값을 기준으로 정렬한다. 현재 snapshot처럼 `forward`가 없으면 기존 spot-moneyness 열은 참고용으로 보존하되, 새로운 forward convention 없이 strike를 x축 변수로 사용해 정렬한다. 실패 행도 `iv_skew_df`에 그대로 남는다.

In [5]:
if "forward" in spx_iv_df.columns:
    valid_forward = (
        np.isfinite(spx_iv_df["forward"]) & (spx_iv_df["forward"] > 0.0)
    )
    log_forward_moneyness = pd.Series(np.nan, index=spx_iv_df.index)
    log_forward_moneyness.loc[valid_forward] = np.log(
        spx_iv_df.loc[valid_forward, "strike"]
        / spx_iv_df.loc[valid_forward, "forward"]
    )
    spx_iv_df["log_forward_moneyness"] = log_forward_moneyness

tidy_columns = ["strike", "option_type", "market_price"]
tidy_columns.extend(
    column
    for column in (
        "forward",
        "spot_moneyness",
        "log_spot_moneyness",
        "log_forward_moneyness",
    )
    if column in spx_iv_df.columns
)
tidy_columns.extend(
    [
        "implied_volatility",
        "repricing_error",
        "iv_status",
        "iv_failure_reason",
    ]
)

x_axis_column = (
    "log_forward_moneyness"
    if "log_forward_moneyness" in spx_iv_df.columns
    else "strike"
)
iv_skew_df = (
    spx_iv_df.loc[:, tidy_columns]
    .sort_values([x_axis_column, "option_type"], kind="stable")
    .reset_index(drop=True)
)
iv_skew_df.head(10)

,strike,option_type,market_price,spot_moneyness,log_spot_moneyness,implied_volatility,repricing_error,iv_status,iv_failure_reason
0,2400,call,5175.85,0.316940,-1.149043,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
1,2600,call,4976.95,0.343352,-1.069000,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
2,2800,call,4777.30,0.369763,-0.994892,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
3,3000,call,4579.00,0.396175,-0.925899,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
4,3000,put,0.10,0.396175,-0.925899,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
5,3200,call,4378.60,0.422587,-0.861361,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
6,3400,call,4180.25,0.448998,-0.800736,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
7,3400,put,0.10,0.448998,-0.800736,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
8,3600,call,3981.30,0.475410,-0.743578,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
9,3600,put,0.15,0.475410,-0.743578,NaN,NaN,failed,"MISSING_MODEL_INPUT:risk_free_rate,dividend_yield"
